In [ ]:
import os
import numpy as np
from tqdm import tqdm
from rdflib import Graph, Literal, Namespace, RDF, RDFS, URIRef, XSD
from elasticsearch import Elasticsearch
from sentence_transformers import SentenceTransformer

INDEX_NAME = os.getenv("INDEX_NAME")

In [ ]:
def create_kg(df, embed_model):
    EMBEDDING_SIZE = 384
    MAPPING = {
        "properties": {
            "id":   {"type": "integer"},
            "value_name": {"type": "text"},
            "value_type": {"type": "text"},
            "value_embedding": {
                "type": "dense_vector",
                "dims": EMBEDDING_SIZE,
                "index": True,
                "similarity": "cosine",
            }
        }
    }

    ENT_INDEX_NAME = f"{INDEX_NAME}_entities_index"
    es_client = Elasticsearch('http://localhost:9200')
    es_client.indices.create(index=ENT_INDEX_NAME, mappings=MAPPING)

    BASE = "https://example.com/political-kg/"

    ENTITY = Namespace(f"{BASE}entity/")
    PREDICATE = Namespace(f"{BASE}predicate/")
    ASSERTION = Namespace(f"{BASE}assertion/")
    PROPERTY = Namespace(f"{BASE}property/")

    graph = Graph()

    graph.bind("entity", ENTITY)
    graph.bind("predicate", PREDICATE)
    graph.bind("rdf", RDF)
    graph.bind("rdfs", RDFS)

    entity_embeddings = {}
    relation_embeddings = {}

    for _, row in tqdm(df.iterrows(), total=len(df)):

        subject = row["subject"]
        object_ = row["object"]
        predicate = row["predicate"]

        subject_label = subject.replace(" ", "_")
        predicate_label = predicate.replace(" ", "_")
        object_label = object_.replace(" ", "_")

        speech_id = row["speech_id"]
        fragment_start = row["start"]
        fragment_end = row["end"]
        fragment_date = row["date"]

        subject_uri = URIRef(f"{ENTITY}{subject_label}")
        predicate_uri = URIRef(f"{PREDICATE}{predicate_label}")
        object_uri = URIRef(f"{ENTITY}{object_label}")

        statement = URIRef(f"{ASSERTION}/{speech_id}_{fragment_start}_{fragment_end}_{np.random.randint(100000)}")

        if subject not in entity_embeddings:
            embedding = embed_model.encode(
                subject,
                normalize_embeddings=True
            )
            entity_embeddings[subject] = embedding

        if object_ not in entity_embeddings:
            embedding = embed_model.encode(
                object_,
                normalize_embeddings=True
            )
            entity_embeddings[object_] = embedding


        if predicate not in relation_embeddings:
            embedding = embed_model.encode(
                predicate,
                normalize_embeddings=True
            )

            relation_embeddings[predicate] = embedding

        graph.add((statement, RDF.type, RDF.Statement))
        graph.add((statement, RDF.subject, subject_uri))
        graph.add((statement, RDF.predicate, predicate_uri))
        graph.add((statement, RDF.object, object_uri))
        graph.add((statement, PROPERTY.speech_id, Literal(speech_id)))
        graph.add((statement, PROPERTY.start, Literal(int(fragment_start))))
        graph.add((statement, PROPERTY.end, Literal(int(fragment_end))))
        graph.add((statement, PROPERTY.date, Literal(fragment_date, datatype=XSD.date)))

    for entity, embedding in entity_embeddings.items():
        entity_doc = {
            "id": hash(entity) % (10 ** 8),
            "value_name": entity,
            "value_type": "entity",
            "value_embedding": embedding.tolist()
        }
        es_client.index(index=ENT_INDEX_NAME, id=entity_doc["id"], document=entity_doc)

    for relation, embedding in relation_embeddings.items():
        relation_doc = {
            "id": hash(relation) % (10 ** 8),
            "value_name": relation,
            "value_type": "relation",
            "value_embedding": embedding.tolist()
        }
        es_client.index(index=ENT_INDEX_NAME, id=relation_doc["id"], document=relation_doc)
    return graph

In [ ]:
embed_model = SentenceTransformer("all-MiniLM-L6-v2", device=os.getenv("MODEL_DEVICE", "cpu"))

g = create_kg(new_triplets, embed_model)
g.serialize(destination=f"{INDEX_NAME}.ttl", format="turtle")